# Reproduce ChakraTS on fev-bench

Reruns [fev-bench](https://huggingface.co/spaces/autogluon/fev-leaderboard) for [ChakraTS](https://huggingface.co/yhatlabs/ChakraTS) with the benchmark's own evaluation code, then compares
each result with the submitted one in this folder.

- `MODE = "quick"` (default): the first 3 tasks, a few minutes.
- `MODE = "full"`: all 100 tasks through the API.

ChakraTS is served through the YHat Labs API: set `YHAT_API_KEY` (create one at https://yhatlabs.com/models/signup). A full run uses credits on your account. The API always serves the current ChakraTS, so if it has been updated since these results were submitted, the rerun can differ from them.

In [ ]:
MODE = "quick"  # "quick" (3 tasks) or "full" (all 100)
FEV_COMMIT = "43b946826813997dd79a4a1ec4cd992a920e1455"  # autogluon/fev with this model's wrapper

In [ ]:
import os, subprocess, sys, urllib.request
HERE = os.getcwd()
if not os.path.exists("fev"):
    subprocess.run(["git", "clone", "-q", "https://github.com/autogluon/fev"], check=True)
subprocess.run(["git", "-C", "fev", "checkout", "-q", FEV_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./fev"], check=True)
if not os.path.exists("chakrats_api.csv"):  # in Colab: the submitted file from GitHub
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yhatlabs/yhatlabs-cookbook/main/benchmarks/fev_bench/ChakraTS/chakrats_api.csv", "chakrats_api.csv")

In [ ]:
import getpass
if not os.environ.get("YHAT_API_KEY"):
    os.environ["YHAT_API_KEY"] = getpass.getpass("YHAT_API_KEY: ")

In [ ]:
os.makedirs("rerun", exist_ok=True)
cmd = [sys.executable, "../fev/models/evaluate.py", "-m", "chakrats_api", "-b", "../fev/benchmarks/fev_bench/tasks.yaml", "-n", "ChakraTS", "-d", "-k", '{"model": "chakra-ts", "preset": "pro"}']
subprocess.run(cmd + (["-t", "3"] if MODE == "quick" else []), cwd="rerun", check=True)

In [ ]:
import pandas as pd
submitted = pd.read_csv("chakrats_api.csv").set_index("task_name")
rerun = pd.read_csv("rerun/ChakraTS.csv").set_index("task_name")
table = pd.DataFrame({"submitted": submitted.loc[rerun.index, "test_error"], "rerun": rerun["test_error"]})
table["difference %"] = 100 * (table["rerun"] / table["submitted"] - 1)
print(table.round(4).to_string())
print(f"{len(table)} tasks, largest difference {table['difference %'].abs().max():.3f}%")